In [1]:
import pandas as pd
df = pd.read_csv('train.csv')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.impute import SimpleImputer
# feature engineer

def engineered_features(df):
    """針對 Top 重要特徵進行非線性交互作用與缺失模式診斷的特徵工程。

    參考依據 (Feature Importance Gain 分析與數據洞察)：
    1. 變數貢獻高度集中：
    - stress_level 與 physical_activity 貢獻了絕大部分的預測增益。

    2. 缺失值的資訊量 (Informative Missingness) 發現：
    - 在預處理實驗過程中，觀察到原始資料帶有 NaN 狀態的特徵（如 stress_level_nan 351.9、
      physical_activity_nan 247.2）重要性顯著超越 BMI、心率等傳統生理指標。
    - 證實此資料集的缺失並非隨機發生 (MNAR)，「未填寫/資料遺漏」本身即帶有強烈的風險行為特徵。

    工程策略：
    1. 類別交互特徵 (Interaction Feature)：
    - 建立 stress x activity 交叉組合，手動建構非線性情境（例如高壓力+低運動 vs 高壓力+高運動）。
      因根據邏輯來看運動可產生腦內啡，可是十運動時，大腦會釋放腦內啡。它能結合大腦中的阿片受體（Opioid receptors），
      抑制疼痛訊號，同時帶來愉悅與放鬆感，抵銷高壓帶來的焦慮情緒。
    
    2. 缺失模式
    - 針對三大核心指標 (sleep, stress, activity)的缺失值。衍生出「缺失數量」與「缺失型態」，
      表示受測者的問卷填寫行為模式。
    """
    df = df.copy()

    # 1. 壓力與活動量的交互作用
    df['stress_activity_interaction'] = (
        df["stress_level"].astype(str)
      + '_'
      + df["physical_activity_level"].astype(str)
    )

    # 2. 缺失數量
    missing_sleep = df["sleep_duration"].isna().astype(int)
    missing_stress = df['stress_level'].isna().astype(int)
    missing_activity = df['physical_activity_level'].isna().astype(int)
    df['missing_key_counts'] = missing_sleep + missing_stress +missing_activity

    # 3. 缺失型態
    df['missing_pattern'] = (
        missing_sleep.astype(str)
        + '_'
        + missing_stress.astype(str)
        + '_'
        + missing_activity.astype(str)
    )
    return df

# num_transformer 
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# key_cat_transformer(保留關鍵類別的遺漏訊號)
key_cat_trasformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value = 'missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# others_cat_transformer
others_cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 欄位轉換
key_cat_cols = [
    'stress_level',
    'physical_activity_level',
    'stress_activity_interaction',
    'missing_pattern',
]
other_cat_cols = [
    'diet_type',
    'smoking_alcohol',
    'gender'
]
column_trans =  ColumnTransformer(
    transformers = [
        ('num', num_transformer, make_column_selector(dtype_include=['number'])),
        ('key_cat', key_cat_trasformer, key_cat_cols),
        ('others_cat', others_cat_transformer, other_cat_cols)
])


# 整合資料處理管線
preprocessor = Pipeline(steps=[
    ('feature_engineer', FunctionTransformer(engineered_features)),
    ('column_transformer', column_trans)
])

preprocessor.set_output(transform='pandas')

,steps,"[('feature_engineer', ...), ('column_transformer', ...)]"
,transform_input,None
,memory,None
,verbose,False
,func,<function eng...00215EC28EDD0>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [27]:
# define the data
from sklearn.preprocessing import LabelEncoder
# X
X = df.loc[:, [c for c in df.columns if c not in ['id', 'health_condition']]]
# y
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['health_condition'])
# split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
# 確認一下資料
test = preprocessor.fit_transform(X)
print(test.isna().sum().sum(),test.shape) 
test.info() 


0 (690088, 49)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 49 columns):
 #   Column                                                 Non-Null Count   Dtype  
---  ------                                                 --------------   -----  
 0   num__sleep_duration                                    690088 non-null  float64
 1   num__heart_rate                                        690088 non-null  float64
 2   num__bmi                                               690088 non-null  float64
 3   num__calorie_expenditure                               690088 non-null  float64
 4   num__step_count                                        690088 non-null  float64
 5   num__exercise_duration                                 690088 non-null  float64
 6   num__water_intake                                      690088 non-null  float64
 7   num__missing_key_counts                                690088 non-null  float64
 8   key_cat__stress_lev

In [35]:
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
fina_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGB', XGBClassifier(
        n_estimators = 459,
        learning_rate = 0.05,
        max_depth = 7,
        min_child_weight = 1, 
        subsample = 0.8,
        colsample_bytree = 1.0,
        gamma = 0, 
        reg_alpha = 0,
        reg_lambda = 1,
        random_state=42,
        tree_method = 'hist',
        n_jobs=-1
    ))    
])

fina_model.fit(X_train, y_train, XGB__sample_weight=np.sqrt(compute_sample_weight('balanced',  y_train)))

,steps,"[('preprocessor', ...), ('XGB', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('feature_engineer', ...), ('column_transformer', ...)]"
,transform_input,None
,memory,None
,verbose,False
,func,<function eng...00264CA521C60>
,inverse_func,None
,validate,False


In [36]:
from sklearn.metrics import classification_report
fina_model_train_predict = fina_model.predict(X_train)
fina_model_test_predict = fina_model.predict(X_test)
report_fina_model_train = classification_report(y_train, fina_model_train_predict)
report_fina_model_test = classification_report(y_test,fina_model_test_predict)
print(f'train: {report_fina_model_train}\n test: {report_fina_model_test}')

train:               precision    recall  f1-score   support

           0       0.99      0.96      0.98    473940
           1       0.81      0.95      0.87     31844
           2       0.80      0.95      0.87     46286

    accuracy                           0.96    552070
   macro avg       0.87      0.95      0.91    552070
weighted avg       0.97      0.96      0.96    552070

 test:               precision    recall  f1-score   support

           0       0.99      0.96      0.97    118621
           1       0.79      0.93      0.85      7959
           2       0.77      0.92      0.84     11438

    accuracy                           0.95    138018
   macro avg       0.85      0.94      0.89    138018
weighted avg       0.96      0.95      0.95    138018



In [41]:
import pandas as pd

# 1. 直接用原本轉換好的 DataFrame 欄位名稱，或者現場 transform 出欄位名
feature_names = preprocessor.transform(X_train.head(1)).columns
importances = fina_model.named_steps['XGB'].feature_importances_

# 2. 組成 DataFrame 並排序
imp_df = pd.DataFrame(
    {'feature': feature_names, 'importance': importances}
).sort_values('importance', ascending=False)

# 3. 印出前 20 大特徵
imp_df['clean_feature'] = imp_df['feature'].str.split('__').str[-1]
display(imp_df[['clean_feature', 'importance']].head(20))

,clean_feature,importance
20,stress_activity_interaction_low_active,0.425536
10,stress_level_medium,0.111784
8,stress_level_high,0.091753
28,stress_activity_interaction_nan_active,0.067133
22,stress_activity_interaction_low_nan,0.053210
9,stress_level_low,0.046919
11,stress_level_missing,0.044266
36,missing_pattern_1_0_0,0.042381
0,sleep_duration,0.024011
34,missing_pattern_0_1_0,0.014734


In [44]:
#　嘗試降低學習率，提升精準度同時計算最佳的n_estimators
learning_rates = [0.1, 0.05, 0.02, 0.01]
# 設定驗證集
X_train_sub, X_train_val, y_train_sub, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)
X_train_sub_proc = preprocessor.fit_transform(X_train_sub)
X_train_val_proc = preprocessor.transform(X_train_val)
X_test_proc = preprocessor.transform(X_test)
from xgboost import XGBClassifier
results = {}
for lr in learning_rates:
    print(f'\n==============learning rate {lr}===================')
    mod2 = XGBClassifier( 
            n_estimators = 3000,
            learning_rate = lr,
            max_depth = 7, 
            min_child_weight = 1, 
            subsample = 0.8,
            colsample_bytree = 1.0,
            gamma = 0, 
            reg_alpha = 0,
            reg_lambda = 1,
            early_stopping_rounds = 30,
            eval_metric="mlogloss",
            random_state=42,

            tree_method = 'hist',
            n_jobs = -2
    )
    mod2.fit(
        X_train_sub_proc, y_train_sub,
        sample_weight = np.sqrt(compute_sample_weight('balanced', y_train_sub)),
        eval_set = [(X_train_val_proc, y_val)],
        sample_weight_eval_set=[np.sqrt(compute_sample_weight('balanced', y_val))],
        verbose = 100
    )
    best_tree = mod2.best_iteration
    # test
    mod2_predict = mod2.predict(X_test_proc)
    mod2_report = classification_report(y_test,mod2_predict)
    results[lr] = {
        'learning rate': lr,
        'n_estimators': best_tree,
        'report': mod2_report
    }


==============learning rate 0.1===================
[0]	validation_0-mlogloss:0.78465
[100]	validation_0-mlogloss:0.14994
[163]	validation_0-mlogloss:0.14952

==============learning rate 0.05===================
[0]	validation_0-mlogloss:0.84406
[100]	validation_0-mlogloss:0.15386
[200]	validation_0-mlogloss:0.14963
[269]	validation_0-mlogloss:0.14932

==============learning rate 0.02===================
[0]	validation_0-mlogloss:0.88099
[100]	validation_0-mlogloss:0.21569
[200]	validation_0-mlogloss:0.15917
[300]	validation_0-mlogloss:0.15197
[400]	validation_0-mlogloss:0.15039
[500]	validation_0-mlogloss:0.14969
[600]	validation_0-mlogloss:0.14933
[683]	validation_0-mlogloss:0.14924

==============learning rate 0.01===================
[0]	validation_0-mlogloss:0.89350
[100]	validation_0-mlogloss:0.35221
[200]	validation_0-mlogloss:0.21675
[300]	validation_0-mlogloss:0.17377
[400]	validation_0-mlogloss:0.15929
[500]	validation_0-mlogloss:0.15403
[600]	validation_0-mlogloss:0.15201
[700]

In [45]:
for lr, info in results.items():
    print("=" * 65),
    print(
        f"🚀 Learning Rate: {lr:<5} | 🌲 Best n_estimators: {info['n_estimators']} 棵樹"
    ),
    print("=" * 65)
    # 印出原本完整的 classification_report 表格
    print(info["report"])
    print("\n")

🚀 Learning Rate: 0.1   | 🌲 Best n_estimators: 133 棵樹
              precision    recall  f1-score   support

           0       0.99      0.95      0.97    118621
           1       0.78      0.93      0.85      7959
           2       0.76      0.93      0.84     11438

    accuracy                           0.95    138018
   macro avg       0.84      0.94      0.89    138018
weighted avg       0.96      0.95      0.95    138018



🚀 Learning Rate: 0.05  | 🌲 Best n_estimators: 239 棵樹
              precision    recall  f1-score   support

           0       0.99      0.95      0.97    118621
           1       0.78      0.93      0.85      7959
           2       0.75      0.93      0.83     11438

    accuracy                           0.95    138018
   macro avg       0.84      0.94      0.88    138018
weighted avg       0.96      0.95      0.95    138018



🚀 Learning Rate: 0.02  | 🌲 Best n_estimators: 653 棵樹
              precision    recall  f1-score   support

           0       0

In [7]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np 
print('1. 開始處理 X 與 y...')
# X
X = df.loc[:, [c for c in df.columns if c not in ['id', 'health_condition']]]
# y
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['health_condition'])
print('2. 進行兩次 train_test_split...')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
X_tr_proc = preprocessor.fit_transform(X_tr) #訓練
X_val_proc = preprocessor.transform(X_val) # 驗證
X_test_proc = preprocessor.transform(X_test)  # 測試
train_weights = np.sqrt(compute_sample_weight('balanced', y_tr))


1. 開始處理 X 與 y...
2. 進行兩次 train_test_split...


In [8]:
# 嘗試貝式優化
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
# 設定參數
def objective(trial, X=X, y=y):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.15),
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, 100),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0 ),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'gamma': trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        'random_state': 42,
        'tree_method': 'hist',
        'n_jobs': -1
    }

    clf = XGBClassifier(**params)
    clf.fit(X_tr_proc, y_tr, sample_weight=train_weights)
    preds = clf.predict(X_val_proc)
    return f1_score(y_val, preds, average='macro')

In [10]:
%%time
import optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=25)

[I 2026-07-26 18:26:57,791] A new study created in memory with name: no-name-abcfaeeb-d2bf-4aac-aae8-15ecb4ebf4b2
C:\Users\user\AppData\Local\Temp\ipykernel_12180\1732640444.py:8: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'n_estimators': trial.suggest_int('n_estimators', 500, 2000, 100),
[I 2026-07-26 18:32:00,714] Trial 0 finished with value: 0.8908480184554991 and parameters: {'learning_rate': 0.07697222624493437, 'n_estimators': 1900, 'max_depth': 4, 'subsample': 0.5465898042088249, 'colsample_bytree': 0.977870749661545, 'min_child_weight': 8, 'alpha': 8.204534343992877e-06, 'lambda': 1.5263763

CPU times: total: 5h 11min 1s
Wall time: 1h 28min 38s


In [11]:
print('Number of finished trials:', len(study.trials))
print('Best trial parameters:', study.best_trial.params)
print('Best score:', study.best_value)

Number of finished trials: 25
Best trial parameters: {'learning_rate': 0.07627551043658595, 'n_estimators': 2000, 'max_depth': 8, 'subsample': 0.8486182392354898, 'colsample_bytree': 0.6822109109092016, 'min_child_weight': 3, 'alpha': 0.11324572621071657, 'lambda': 0.002863397433947658, 'gamma': 2.366003111770322e-07}
Best score: 0.8999031514105985


In [14]:
import numpy as np
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

best_params = study.best_trial.params.copy()


import numpy as np
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGB', XGBClassifier(**best_params))    
])

final_model.fit(X_train, y_train, XGB__sample_weight=np.sqrt(compute_sample_weight('balanced',  y_train)))


,steps,"[('preprocessor', ...), ('XGB', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('feature_engineer', ...), ('column_transformer', ...)]"
,transform_input,None
,memory,None
,verbose,False
,func,<function eng...00215EC28EDD0>
,inverse_func,None
,validate,False


In [15]:
from sklearn.metrics import classification_report
final_model_train_predict = final_model.predict(X_train)
final_model_test_predict = final_model.predict(X_test)
report_final_model_train = classification_report(y_train, final_model_train_predict)
report_final_model_test = classification_report(y_test,final_model_test_predict)
print(f'train: {report_final_model_train}\n test: {report_final_model_test}')

train:               precision    recall  f1-score   support

           0       1.00      1.00      1.00    474049
           1       0.99      1.00      1.00     31842
           2       1.00      1.00      1.00     46179

    accuracy                           1.00    552070
   macro avg       1.00      1.00      1.00    552070
weighted avg       1.00      1.00      1.00    552070

 test:               precision    recall  f1-score   support

           0       0.98      0.98      0.98    118512
           1       0.87      0.87      0.87      7961
           2       0.87      0.85      0.86     11545

    accuracy                           0.96    138018
   macro avg       0.91      0.90      0.90    138018
weighted avg       0.96      0.96      0.96    138018



In [16]:
import numpy as np
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import joblib

final_full_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGB', XGBClassifier(**best_params))    
])

final_full_model.fit(X, y, XGB__sample_weight=np.sqrt(compute_sample_weight('balanced',  y)))
joblib.dump(final_full_model, 'health_xgb_mod2.pkl')

['health_xgb_mod2.pkl']

In [17]:
import json

# 1. 取得最佳超參數字典
best_params = study.best_trial.params

# 2. 存入 JSON 檔案
with open('best_params.json', 'w', encoding='utf-8') as f:
  json.dump(best_params, f, indent=4)

print('🎉 最佳超參數已成功儲存至 best_params.json！')

🎉 最佳超參數已成功儲存至 best_params.json！


In [19]:
import pandas as pd

# 1. 取得 XGBoost 模型的特徵重要性
importance = final_model.named_steps['XGB'].feature_importances_

# 2. 直接拿轉換後的 X_train 欄位名稱 (或是輸入給 XGBoost 的欄位)
# 假設你傳進 Pipeline 的 X_train 欄位名稱是 X_train.columns
try:
  # 如果 preprocessor 處理後還是 DataFrame
  feature_names = final_model.named_steps['preprocessor'].transform(
      X_train.head(1)
  ).columns
except AttributeException:
  # 如果預處理出來是 numpy array，直接用原始 X_train 的欄位名稱
  feature_names = X_train.columns

# 3. 組合並排序
df_imp = (
    pd.DataFrame({'feature': feature_names, 'importance': importance})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

print('=== 特徵重要性 Top 15 ===')
print(df_imp.head(15))

print('=== 特徵重要性 Bottom 10 ===')
print(df_imp.tail(10))

=== 特徵重要性 Top 15 ===
                                              feature  importance
0     key_cat__stress_activity_interaction_low_active    0.265237
1                        key_cat__stress_level_medium    0.129931
2                          key_cat__stress_level_high    0.114245
3                           key_cat__stress_level_low    0.072308
4             key_cat__physical_activity_level_active    0.061553
5        key_cat__stress_activity_interaction_low_nan    0.045948
6                       key_cat__stress_level_missing    0.032702
7     key_cat__stress_activity_interaction_nan_active    0.027788
8   key_cat__stress_activity_interaction_low_seden...    0.024306
9                                 num__sleep_duration    0.021252
10         key_cat__physical_activity_level_sedentary    0.015613
11                     key_cat__missing_pattern_0_0_0    0.015229
12  key_cat__stress_activity_interaction_low_moderate    0.015083
13           key_cat__physical_activity_level_missing  